## Services, not containers

The Swarm primitive isn't a container — it's a **service**, a *declaration*:

> "I want N replicas of image X across the cluster, with these networks, secrets, env, resources, and update policy. Make it so."

Swarm decomposes a service into **tasks** (one per replica), schedules each task onto a node, and creates a container to run it. The container is just an implementation detail — you reason at the service level.

```bash
docker service create --replicas 3 --name web -p 8080:80 nginx:alpine
docker service ls              # web  replicated  3/3  nginx:alpine  *:8080->80
docker service ps web          # the individual tasks + which node each is on
```

The payoff is **self-healing**: if a node hosting a task crashes, the manager notices, reschedules that task elsewhere, and `3/3` is restored automatically — no human paged. That reconciliation loop is the entire point of an orchestrator.

**Two service modes:** **replicated** (default — N tasks, the scheduler picks where; the 90% case) and **global** (exactly one task per node — for per-node agents like log shippers or monitoring).

The service is also the unit of every operation — the single-host verbs move up a level:

| Command | Does |
|---|---|
| `service create` | declare a service (most `docker run` flags apply) |
| `service ls` / `ps` | desired-vs-running / per-task placement |
| `service scale web=5` | change replica count |
| `service update` | change the definition → **rolling update** (next) |
| `service rollback` | revert the last update |
| `service rm` | delete it and all its tasks |

The mental shift: **you don't `stop`/`start` a service — you `scale` it (to 0 to "stop") or `update` it.** State is desired, not imperative.